# `06_bicycle_route_m_ways_distinct_classified_per_spatial_unit`: Classified bicycle route ways per spatial unit

## Introduction

### Purpose

This notebook spatially joins the classified bicycle route ways (`bicycle_route_m_ways_distinct_classified` from `04_bicycle_route_m_ways_distinct_classified`) to the three spatial-unit types established in `03_boundaries_population` (municipalities, provinces, H3 cells). For each way–unit pair, the length of the way portion falling within the unit is computed in EPSG:28992 and stored as `clipped_length_meters`. All classification columns produced in notebook 04 (mapping style, per-side infrastructure labels, way-level infrastructure, access signals, routing surface, directionality, `infrastructure_confidence`) ride through the join unchanged. Per the thesis (§3.4.6), this is the stage-06 spatial-join step that produces the per-spatial-unit tables consumed by the stage-07 metric notebooks.

### Inputs

- `bicycle_route_m_ways_distinct_classified` from `04_bicycle_route_m_ways_distinct_classified` (181,318 ways totalling 38,570.854 km, with the full Step-1-to-Step-9 column set).
- `municipalities`, `provinces`, `h3_cells` from `03_boundaries_population`.
- `clip_ways_to_spatial_unit` helper from `functions.ipynb`.

### Outputs

- `bicycle_route_m_ways_distinct_classified_per_municipality`
- `bicycle_route_m_ways_distinct_classified_per_province`
- `bicycle_route_m_ways_distinct_classified_per_h3`

Each row is one way–unit pair, carrying all original way attributes and classification columns plus `clipped_length_meters`.

### Key steps

For each spatial-unit type, `clip_ways_to_spatial_unit` is called once with the relevant Arrow table. The function computes the geometric intersection of each way with every unit it touches and the length of that intersection in EPSG:28992, a projected CRS that preserves distances in metres across the Netherlands. The clipped geometry itself is not retained; only the length is stored, since the original way geometry is already available in `bicycle_route_m_ways_distinct_classified` and the clipped length is the quantity needed downstream.

A way lying entirely within a single spatial unit produces one row with `clipped_length_meters` equal to its full length. A way crossing multiple spatial units produces one row per intersected unit, with `clipped_length_meters` reflecting the length of each portion. In both cases the sum of `clipped_length_meters` across all rows for a given way equals its original total length, except where boundary geometry causes small losses (visible in the H3 row of the validation table, which sums to 38,482.700 km against the input's 38,570.854 km).

### Dependencies on prior notebooks

- `04_bicycle_route_m_ways_distinct_classified.ipynb`: provides `bicycle_route_m_ways_distinct_classified` (and transitively brings in `00_bicycle_route_relations` and `03_boundaries_population` via its own `%run`).
- `03_boundaries_population.ipynb`: provides `municipalities`, `provinces`, `h3_cells` (re-run here explicitly to make the dependency surface visible).
- `functions.ipynb`: provides the `clip_ways_to_spatial_unit` helper used in all three sections.

### Downstream consumers

Per the pipeline diagram,

## 1. Environment setup

### Libraries and extensions

In [2]:
from IPython.utils import io

### Loading shared variables

Execute the prior notebooks to bring `bicycle_route_m_ways_distinct_classified`, the three boundary-and-population tables (`municipalities`, `provinces`, `h3_cells`), and the `clip_ways_to_spatial_unit` helper into the current session.

In [9]:
with io.capture_output() as captured:
    %run /home/vbo226/04_bicycle_route_m_ways_distinct_classified.ipynb
    %run /home/vbo226/03_boundaries_population.ipynb
    %run /home/vbo226/functions.ipynb  # for function 

In [10]:
# For each spatial unit (municipality, province and h3 cell), convert GeoDataFrame into Arrow for easy ingestion into DuckDB 
municipality_arrow = municipalities.to_arrow()
provinces_arrow = provinces.to_arrow()
h3_cells_arrow = h3_cells.to_arrow()

---

## 2. Core clipping function
 
`clip_ways_to_spatial_unit` is defined in `06_non_bicycle_route_ways_per_spatial_unit` and reused here without modification. Running that notebook imports the function into the current session.

## 3. Municipalities

Clip the bicycle route ways to the 342 Dutch municipalities. Each way produces one row per municipality it intersects, with `clipped_length_meters` set to the within-municipality length.

In [21]:
# Clip member ways to municipal areas 
bicycle_route_m_ways_distinct_classified_per_municipality = clip_ways_to_spatial_unit(
    spatial_unit_table='municipality_arrow',
    ways_table='bicycle_route_m_ways_distinct_classified',
    geom_col='way_m_geom'
)

# Row count after clipping
bicycle_route_m_ways_distinct_per_municipality.count('*')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       185117 │
└──────────────┘

## 4. Provinces

Clip the bicycle route ways to the 12 Dutch provinces, producing one row per way–province intersection.

In [19]:
# Clip member ways to provincial areas 
bicycle_route_m_ways_distinct_classified_per_province = clip_ways_to_spatial_unit(
    spatial_unit_table='provinces_arrow',
    ways_table='bicycle_route_m_ways_distinct_classified',
    geom_col='way_m_geom'
)

# Row count after clipping
bicycle_route_m_ways_distinct_per_province.count('*')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       181774 │
└──────────────┘

## 5. H3 grid cells

Clip the bicycle route ways to the 66,530 resolution-8 H3 cells covering the Netherlands. Because the H3 grid is much finer than the administrative boundaries, the number of way–cell rows is substantially larger than the input row count.

In [18]:
# Clip member ways to H3 cell areas 
bicycle_route_m_ways_distinct_classified_per_h3 = clip_ways_to_spatial_unit(
    spatial_unit_table='h3_cells_arrow',
    ways_table='bicycle_route_m_ways_distinct',
    geom_col='way_m_geom'
)

# Row count after clipping
bicycle_route_m_ways_distinct_per_h3_cell.count('*')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       237779 │
└──────────────┘

## 6. Validation

To validate the spatial intersection, the number of resulting rows is compared against the original distinct-way count, and the sum of `clipped_length_meters` is compared against the total length in `bicycle_route_m_ways_distinct_classified`. Both checks should hold: row counts will exceed the original way count where ways cross spatial-unit boundaries, but the total clipped length should equal the total original length.

In [16]:
original_count = duckdb.sql("""
    SELECT COUNT(*) FROM bicycle_route_m_ways_distinct_classified
""").fetchone()[0]
 
original_length_km = duckdb.sql("""
    SELECT ROUND(SUM(length_nl_meters) / 1000, 3) FROM bicycle_route_m_ways_distinct_classified
""").fetchone()[0]
 
municipality_count  = bicycle_route_m_ways_distinct_per_municipality.count('*').fetchone()[0]
province_count      = bicycle_route_m_ways_distinct_per_province.count('*').fetchone()[0]
h3_count            = bicycle_route_m_ways_distinct_per_h3_cell.count('*').fetchone()[0]
 
municipality_length_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM bicycle_route_m_ways_distinct_per_municipality
""").fetchone()[0]
 
province_length_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM bicycle_route_m_ways_distinct_per_province
""").fetchone()[0]
 
h3_length_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM bicycle_route_m_ways_distinct_per_h3_cell
""").fetchone()[0]
 
print(f"{'':40s} {'rows':>10s}  {'multiplier':>10s}  {'length (km)':>12s}")
print(f"{'─'*76}")
print(f"{'Original (bicycle_route_m_ways_distinct_classified)':40s} {original_count:>10,}  {'':>10s}  {original_length_km:>12,.3f}")
print(f"{'After municipality join':40s} {municipality_count:>10,}  {municipality_count/original_count:>10.3f}x  {municipality_length_km:>12,.3f}")
print(f"{'After province join':40s} {province_count:>10,}  {province_count/original_count:>10.3f}x  {province_length_km:>12,.3f}")
print(f"{'After H3 join':40s} {h3_count:>10,}  {h3_count/original_count:>10.3f}x  {h3_length_km:>12,.3f}")

                                               rows  multiplier   length (km)
────────────────────────────────────────────────────────────────────────────
Original (bicycle_route_m_ways_distinct_classified)    181,318                38,570.854
After municipality join                     185,117       1.021x    38,570.854
After province join                         181,774       1.003x    38,570.854
After H3 join                               237,779       1.311x    38,482.700


Row counts will exceed the original way count wherever ways cross spatial unit boundaries; the multiplier reflects this fragmentation. Coarser administrative units introduce limited splitting, while the finer H3 grid induces substantially more fragmentation. Total clipped length should match the original in all three cases, any discrepancy indicates a geometry or projection issue.

### Export

Persist the three per-spatial-unit tables to disk so the stage-07 metrics notebooks can read them without re-running the spatial join.

In [22]:
from pathlib import Path
import os

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)


def safe_write_parquet(df, final_path):
    tmp_path = final_path + ".tmp"

    df.write_parquet(tmp_path)
    os.replace(tmp_path, final_path)

safe_write_parquet(
    bicycle_route_m_ways_distinct_classified_per_municipality,
    "/local/data/vbo226/cache/bicycle_route_m_ways_distinct_classified_per_municipality.parquet"
)

safe_write_parquet(
    bicycle_route_m_ways_distinct_classified_per_province,
    "/local/data/vbo226/cache/bicycle_route_m_ways_distinct_classified_per_province.parquet"
)

safe_write_parquet(
    bicycle_route_m_ways_distinct_classified_per_h3,
    "/local/data/vbo226/cache/bicycle_route_m_ways_distinct_classified_per_h3.parquet"
)